# Phase 3: µP Scaling Study
## SVG Scaling Laws — CS-GY 6923

**Steps:**
1. Mount Drive & pull repo
2. Generate µP base shapes (one-time, fast)
3. LR sweep on Tiny µP (2000 steps × 7 LRs)
4. Update best µP LR in config
5. Train each of the 5 µP models individually
6. Fit scaling law and generate comparison plots (SP vs µP)
7. Scaling law extrapolation

**Prerequisites:** Phase 2 must be complete (all 5 SP results in `outputs/logs/result_*.json`).

---
## Cell 0: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/svg-scaling-laws'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Project dir: {DRIVE_DIR}')

---
## Cell 1: Clone / Pull Repository

In [ ]:
REPO_URL = 'https://github.com/YOUR_USERNAME/svg-scaling-laws.git'  # <-- EDIT THIS
REPO_DIR = '/content/svg-scaling-laws'

import os
if os.path.exists(REPO_DIR):
    print('Pulling latest ...')
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')

---
## Cell 2: Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q mup
print('Dependencies installed.')

---
## Cell 3: Configure Output Paths

In [ ]:
import os, sys

REPO_DIR      = '/content/svg-scaling-laws'
DRIVE_OUTPUTS = '/content/drive/MyDrive/svg-scaling-laws/outputs'
LOCAL_OUTPUTS = os.path.join(REPO_DIR, 'outputs')

os.makedirs(DRIVE_OUTPUTS, exist_ok=True)

if os.path.islink(LOCAL_OUTPUTS):
    os.unlink(LOCAL_OUTPUTS)
elif os.path.exists(LOCAL_OUTPUTS):
    import shutil
    shutil.rmtree(LOCAL_OUTPUTS)

os.symlink(DRIVE_OUTPUTS, LOCAL_OUTPUTS)
print(f'Symlink: {LOCAL_OUTPUTS} -> {DRIVE_OUTPUTS}')

for d in ['logs', 'checkpoints_mup', 'plots']:
    os.makedirs(os.path.join(DRIVE_OUTPUTS, d), exist_ok=True)

sys.path.insert(0, REPO_DIR)
print('Path configured.')

---
## Cell 4: Verify Phase 2 Results Exist

In [ ]:
import json, os

model_names = ['tiny', 'small', 'medium', 'large', 'xl']
print('Phase 2 SP results:')
all_present = True
for name in model_names:
    p = f'outputs/logs/result_{name}.json'
    if os.path.exists(p):
        with open(p) as f:
            r = json.load(f)
        print(f'  {name:<8} N={r["n_params"]:>10,}  val_loss={r["best_val_loss"]:.4f}  OK')
    else:
        print(f'  {name:<8} MISSING — run Phase 2 first!')
        all_present = False

if all_present:
    print('\nAll Phase 2 results found. Ready for Phase 3.')
else:
    print('\nComplete Phase 2 before running Phase 3.')

---
## Cell 5: Generate µP Base Shapes
One-time setup. Creates `outputs/base_shapes.bsh` from Tiny (d=128) and a delta model (d=256).
Takes <30 seconds on CPU. Re-run is safe (overwrites the file).

In [ ]:
!python scripts/07_train_mup.py --make_base_shapes

import os
bsh = 'outputs/base_shapes.bsh'
if os.path.exists(bsh):
    print(f'base_shapes.bsh exists ({os.path.getsize(bsh):,} bytes)')
else:
    print('ERROR: base_shapes.bsh not created!')

---
## Cell 6: µP LR Sweep — Tiny Model
Same 7 LRs as Phase 2, same 2000-step budget.

In [ ]:
!python scripts/08_lr_sweep_mup.py --training_config configs/training_config.yaml

---
## Cell 7: Display µP LR Sweep Results

In [ ]:
import json
from IPython.display import Image, display

with open('outputs/logs/lr_sweep_mup.json') as f:
    mup_sweep = json.load(f)

# Also load SP sweep for comparison
with open('outputs/logs/lr_sweep_sp.json') as f:
    sp_sweep = json.load(f)

print(f"SP  best LR: {sp_sweep['best_lr']:.1e}  →  val_loss={sp_sweep['best_val_loss']:.4f}")
print(f"µP  best LR: {mup_sweep['best_lr']:.1e}  →  val_loss={mup_sweep['best_val_loss']:.4f}")

print(f"\n{'LR':>12} {'SP val':>10} {'µP val':>10}")
print("-" * 36)
sp_by_lr = {r['lr']: r['val_loss'] for r in sp_sweep['runs']}
for r in sorted(mup_sweep['runs'], key=lambda x: x['lr']):
    sp_v = sp_by_lr.get(r['lr'], float('nan'))
    mup_v = r['val_loss']
    marker = '*' if r['lr'] == mup_sweep['best_lr'] else ' '
    print(f"{r['lr']:>12.1e} {sp_v:>10.4f} {mup_v:>10.4f} {marker}")

display(Image('outputs/plots/lr_sweep_comparison.png'))

In [ ]:
# Auto-update training_config.yaml with best µP LR
import yaml, json

with open('outputs/logs/lr_sweep_mup.json') as f:
    mup_sweep = json.load(f)

best_mup_lr = mup_sweep['best_lr']

with open('configs/training_config.yaml') as f:
    tcfg = yaml.safe_load(f)

tcfg['mup_learning_rate'] = best_mup_lr

with open('configs/training_config.yaml', 'w') as f:
    yaml.dump(tcfg, f, default_flow_style=False, allow_unicode=True)

print(f'configs/training_config.yaml updated: mup_learning_rate = {best_mup_lr:.1e}')
print(f'Use --lr {best_mup_lr:.1e} when calling 07_train_mup.py')

---
## Cell 8: Train µP — Tiny

In [ ]:
import json
with open('outputs/logs/lr_sweep_mup.json') as f:
    best_lr = json.load(f)['best_lr']

!python scripts/07_train_mup.py --model_name tiny --lr {best_lr} --resume

In [ ]:
import json
with open('outputs/logs/result_mup_tiny.json') as f:
    r = json.load(f)
print(f"µP Tiny complete.  Val loss: {r['best_val_loss']:.4f}.  Time: {r['wall_time_min']:.1f} min.")

---
## Cell 9: Train µP — Small

In [ ]:
import json
with open('outputs/logs/lr_sweep_mup.json') as f:
    best_lr = json.load(f)['best_lr']

!python scripts/07_train_mup.py --model_name small --lr {best_lr} --resume

In [ ]:
import json
with open('outputs/logs/result_mup_small.json') as f:
    r = json.load(f)
print(f"µP Small complete.  Val loss: {r['best_val_loss']:.4f}.  Time: {r['wall_time_min']:.1f} min.")

---
## Cell 10: Train µP — Medium

In [ ]:
import json
with open('outputs/logs/lr_sweep_mup.json') as f:
    best_lr = json.load(f)['best_lr']

!python scripts/07_train_mup.py --model_name medium --lr {best_lr} --resume

In [ ]:
import json
with open('outputs/logs/result_mup_medium.json') as f:
    r = json.load(f)
print(f"µP Medium complete.  Val loss: {r['best_val_loss']:.4f}.  Time: {r['wall_time_min']:.1f} min.")

---
## Cell 11: Train µP — Large

In [ ]:
import json
with open('outputs/logs/lr_sweep_mup.json') as f:
    best_lr = json.load(f)['best_lr']

!python scripts/07_train_mup.py --model_name large --lr {best_lr} --resume

In [ ]:
import json
with open('outputs/logs/result_mup_large.json') as f:
    r = json.load(f)
print(f"µP Large complete.  Val loss: {r['best_val_loss']:.4f}.  Time: {r['wall_time_min']:.1f} min.")

---
## Cell 12: Train µP — XL
Uses `--grad_accum 2` to halve peak memory.

In [ ]:
import json
with open('outputs/logs/lr_sweep_mup.json') as f:
    best_lr = json.load(f)['best_lr']

!python scripts/07_train_mup.py --model_name xl --lr {best_lr} --grad_accum 2 --resume

In [ ]:
import json
with open('outputs/logs/result_mup_xl.json') as f:
    r = json.load(f)
print(f"µP XL complete.  Val loss: {r['best_val_loss']:.4f}.  Time: {r['wall_time_min']:.1f} min.")

---
## Cell 13: Summary — All µP Models Trained

In [ ]:
import json, os

model_names = ['tiny', 'small', 'medium', 'large', 'xl']
print(f"{'Model':<10} {'Params':>12} {'SP val L':>10} {'µP val L':>10} {'Δ':>8}")
print("-" * 56)

all_done = True
for name in model_names:
    sp_path  = f'outputs/logs/result_{name}.json'
    mup_path = f'outputs/logs/result_mup_{name}.json'
    sp_v  = json.load(open(sp_path))['best_val_loss']  if os.path.exists(sp_path)  else float('nan')
    mup_v = json.load(open(mup_path))['best_val_loss'] if os.path.exists(mup_path) else None
    n_params = json.load(open(sp_path))['n_params'] if os.path.exists(sp_path) else '?'

    if mup_v is None:
        print(f"{name:<10} {n_params:>12,} {sp_v:>10.4f} {'NOT DONE':>10}")
        all_done = False
    else:
        delta = mup_v - sp_v
        arrow = '↓' if delta < 0 else '↑'
        print(f"{name:<10} {n_params:>12,} {sp_v:>10.4f} {mup_v:>10.4f} {delta:>+7.4f}{arrow}")

if all_done:
    print("\nAll 5 µP models trained. Ready to fit scaling law.")
else:
    print("\nSome models still need training.")

---
## Cell 14: Fit Scaling Laws — SP vs µP Comparison

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '/content/svg-scaling-laws')

from src.scaling_law import fit_scaling_law, plot_scaling_law, print_fit_summary, predict

model_names = ['tiny', 'small', 'medium', 'large', 'xl']

# Load SP results
sp_params, sp_losses = [], []
for name in model_names:
    p = f'outputs/logs/result_{name}.json'
    if os.path.exists(p):
        r = json.load(open(p))
        sp_params.append(r['n_params'])
        sp_losses.append(r['best_val_loss'])

# Load µP results
mup_params, mup_losses = [], []
for name in model_names:
    p = f'outputs/logs/result_mup_{name}.json'
    if os.path.exists(p):
        r = json.load(open(p))
        mup_params.append(r['n_params'])
        mup_losses.append(r['best_val_loss'])

# Fit both
sp_fit  = fit_scaling_law(sp_params,  sp_losses)
mup_fit = fit_scaling_law(mup_params, mup_losses)

print_fit_summary(sp_fit,  label='SP')
print_fit_summary(mup_fit, label='µP')

# Overlay plot
fig, ax = plt.subplots(figsize=(9, 5))
plot_scaling_law(sp_params,  sp_losses,  sp_fit,  model_names=model_names,
                 label='SP',  color='steelblue', ax=ax)
plot_scaling_law(mup_params, mup_losses, mup_fit, model_names=model_names,
                 label='µP', color='darkorange', ax=ax)
ax.set_title('Scaling Law: SP vs µP\nL = a · N^(−α) + c')
plt.tight_layout()
plt.savefig('outputs/plots/scaling_law_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to outputs/plots/scaling_law_comparison.png')

# Save fit JSONs
def save_fit(fit, name, params, losses):
    out = {
        'parameterization': name,
        'param_counts': params,
        'val_losses': losses,
        'a': fit['a'], 'alpha': fit['alpha'], 'c': fit['c'],
        'r_squared': fit['r_squared'],
    }
    with open(f'outputs/logs/scaling_fit_{name.lower()}.json', 'w') as f:
        json.dump(out, f, indent=2)

save_fit(sp_fit,  'sp',  sp_params,  sp_losses)
save_fit(mup_fit, 'mup', mup_params, mup_losses)

---
## Cell 15: Scaling Law Extrapolation

In [ ]:
from src.scaling_law import predict

max_n = max(sp_params)
N_10x = max_n * 10

sp_pred  = predict(sp_fit,  N_10x)
mup_pred = predict(mup_fit, N_10x)

print(f'Extrapolation: 10× XL ≈ {N_10x/1e6:.0f}M params')
print()
print(f"SP  predicted L = {sp_pred['L_pred']:.4f}  "
      f"95% CI: [{sp_pred['L_lower']:.4f}, {sp_pred['L_upper']:.4f}]")
print(f"µP  predicted L = {mup_pred['L_pred']:.4f}  "
      f"95% CI: [{mup_pred['L_lower']:.4f}, {mup_pred['L_upper']:.4f}]")
print()
print(f"Better parameterization at 10× scale: {'µP' if mup_pred['L_pred'] < sp_pred['L_pred'] else 'SP'}")

---
## Cell 16: Display All Phase 3 Plots

In [ ]:
from IPython.display import Image, display
import os

plots = [
    ('SP vs µP Scaling Law',      'outputs/plots/scaling_law_comparison.png'),
    ('LR Sweep Comparison',       'outputs/plots/lr_sweep_comparison.png'),
]

for title, path in plots:
    if os.path.exists(path):
        print(f'\n--- {title} ---')
        display(Image(path))
    else:
        print(f'Missing: {path}')

---
## Done!

Phase 3 outputs saved to Drive:

| File | Purpose |
|------|---------|
| `outputs/base_shapes.bsh` | µP base shapes (reused across all models) |
| `outputs/logs/lr_sweep_mup.json` | µP LR sweep results |
| `outputs/logs/result_mup_<model>.json` | Best val loss per µP model |
| `outputs/logs/scaling_fit_mup.json` | Fitted µP power law parameters |
| `outputs/plots/scaling_law_comparison.png` | SP vs µP scaling curves |
| `outputs/plots/lr_sweep_comparison.png` | SP vs µP LR sweep overlay |

**Next:** Phase 4 — Generation and Evaluation (`04_generation.ipynb`)